In [ ]:
import pandas as pd 
df = pd.read_csv("diabetes_dirty_dataset.csv")
df

In [ ]:
#check column names, datatypes
df.info()

In [ ]:
#clean the column headers
df.columns.str.lower().str.strip()


In [ ]:
#make all categorical data in smallcase 
cat_col = df.select_dtypes(include=["str"]).columns
for c in cat_col:
    df[c] = df[c].str.strip().str.lower()
df

In [ ]:
#patient id column cleaning 
df["patient_id"].describe()
id = df["patient_id"].value_counts()  #how many times same thing appears
duplicate_id = id[id > 1]    #count duplicate from calculate
print(duplicate_id)
df = df.drop_duplicates(subset = "patient_id", keep="first")   # lesson - drop for whole df not column 
df["patient_id"].describe()
df["patient_id"].isnull().sum()

In [ ]:
#name column cleaning
df["name"].isnull().sum()

In [ ]:
#cleaning of column "Age"
print("null values", df["age"].isnull().sum())
print(df["age"].dtype)
df["age"].describe()

#got -5 & 250 as impossible values 
#need to check how many impossible values there 
min_age = (df["age"] < 0).sum()       #use sum() always count() gives not null values only
max_age= (df["age"]> 120).sum()
total_age_count = min_age + max_age
print(f"total_outlier_count:{total_age_count}")
#now calculate percenage of it 
pet_outlier_count = (total_age_count/len(df))*100
print(f"percentage of outlier: {pet_outlier_count}%")   # f" all things want to include and then close it "  

#will impute this values with median as they are more then 1%
#median over mean because mean calculate outliers too
#mean = sum of all / no of rows !!  median = take no middle one if odd, if even then sum of two middle 
if pet_outlier_count < 1:
    df = df[(df["age"] <= 0) & (df["age"] >= 120)]
else:
    df.loc[(df["age"] <= 0) | (df["age"] >= 120), "age"] = df["age"].median()
# .loc(rows, columns) this selects the rows and columns at same time 
df["age"].describe()
df["age"].head(10)

In [ ]:
#Normalise and clean Gender Column 
df["gender"].unique()
gender_normalise = {
    "male" : "M",
    "f" : "F",
    "female" : "F",
    "m" : "M"}
df["gender"] = df["gender"].replace(gender_normalise)

df["gender"].isnull().sum()
df["gender"] = df["gender"].fillna("NA")


In [ ]:
#usually height woud be from approx 45 cm (Newborn) to  approx 210 cm (players)
df["height_cm"].describe
df["height_cm"].isnull().sum()
df["height_cm"] = df["height_cm"].fillna(df["height_cm"].median())

In [ ]:
#Weight column 
df["weight_kg"].describe()
# will check 238 kg is real or outlier 
df["weight_kg"].sort_values(ascending= False).head(10)
#check the details 
print(df[df["weight_kg"]== df["weight_kg"].max()])

#check ot again tomorrow 

In [ ]:
df.columns

In [ ]:
#BMI column filled with formula 
df["bmi"].describe()
df["bmi"].isnull().sum()
df["bmi"] = df["weight_kg"]/(df["height_cm"]/100)** 2


In [ ]:
#fasting_glucose_mg_dl
df["fasting_glucose_mg_dl"].isnull().sum()
df["fasting_glucose_mg_dl"] = df["fasting_glucose_mg_dl"].fillna(df["fasting_glucose_mg_dl"].median())

In [ ]:
#Now will impute remaining columns collectively 
already_done = ["age", "height_cm", "weight_kg", "bmi", "fasting_glucose_mg_dl"]
num_col = df.select_dtypes(include = "number").columns
remaining_col = num_col.drop(already_done, errors= "ignore")
print(remaining_col.tolist())
df[remaining_col] = df[remaining_col].fillna(df[remaining_col].median())   #remaining col is created variable so no double quote


In [ ]:
#will impute all categorical columns collectively
cat_col_already_done = ["patient_id", "name", "gender"]
cat_col_remaining = df.select_dtypes(include= "str").columns
cat_remaining_col = cat_col_remaining.drop(cat_col_already_done, errors= "ignore")
print(cat_remaining_col.tolist())
df[cat_remaining_col] = df[cat_remaining_col].fillna("NA")

In [ ]:
#diabetes type
df.diabetes_type.unique()
diabetes_types = {
    "type2" : "type 2",
    "t2" : "type 2"}
df["diabetes_type"] = (df["diabetes_type"].replace(diabetes_types))

In [ ]:
# all descriptive column normalization
for col in cat_remaining_col:
    print(col,":", df[col].unique())

all_mappings = {
    "smoking_status": {
        "never" : "non-smoker",
        "current" : "smoker"} ,
    "family_history_diabetes" :{
        "n" : "no",
        "y" : "yes"}}
for column, mapping in all_mappings.items():
    df[column] = df[column].replace(mapping)

In [ ]:
#date column 
df["diagnosis_date"] = pd.to_datetime((df["diagnosis_date"]),format= "mixed", dayfirst= True)